<a href="https://colab.research.google.com/github/lukalklikadze/walmart-sales-forecasting/blob/main/notebooks/model_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ensemble and Kaggle submission

This notebook does not train anything. Every model notebook already logged a final
pipeline that was refit on the **full** train set and predicts straight on the raw
`test` frame (no manual preprocessing). Here we load several of those pipelines, run each
one on the same raw test frame, and **average their predictions**.

Averaging models from different families cancels their systematic biases: one family tends
to under-forecast, another to over-forecast, so the mean lands closer to the truth. A plain
equal-weight mean is especially robust because there is nothing to tune and therefore
nothing to overfit.

Our single best model (LightGBM) scored **Public 2846 / Private 3013** on Kaggle, and a
first LightGBM+Prophet average already improved that to **Public 2806 / Private 2958**.
Here we sweep every sensible combination and keep the best.

## 1. Bootstrap

Same setup as every other notebook: clone the repo, read the Colab secrets, download the
Kaggle data. We install the libraries every candidate model needs to *unpickle*: LightGBM
and XGBoost for the trees, `neuralforecast`+`torch` for N-BEATS, and `prophet` for Prophet.
If one of those installs is flaky on Colab, only that model drops out of the ensemble; the
rest still load.

In [ ]:
!pip -q install mlflow kaggle lightgbm xgboost neuralforecast prophet

!git clone -q https://github.com/lukalklikadze/walmart-sales-forecasting.git /content/repo \
    2>/dev/null || (cd /content/repo && git pull -q)
import sys; sys.path.append("/content/repo")

from src.config import setup_env, DATA_DIR, KAGGLE_COMP
tracking_uri = setup_env()
print("MLflow \u2192", tracking_uri)

import os, glob, zipfile, subprocess
os.makedirs(DATA_DIR, exist_ok=True)
subprocess.run(["kaggle","competitions","download","-c",KAGGLE_COMP,"-p",DATA_DIR,"--force"], check=True)
with zipfile.ZipFile(os.path.join(DATA_DIR, KAGGLE_COMP + ".zip")) as z: z.extractall(DATA_DIR)
for inner in glob.glob(os.path.join(DATA_DIR, "*.csv.zip")):
    with zipfile.ZipFile(inner) as z: z.extractall(DATA_DIR)

from src.data import load_raw
train, test, stores, features = load_raw()
print("train:", train.shape, "| test:", test.shape)

## 2. Imports and wrapper classes

The calendar/lag transformers live in `src.features`, so importing them is enough for the
tree pipelines. The LightGBM notebook defined two wrapper classes in-notebook; cloudpickle
embeds them in the saved model, but we redeclare them byte-for-byte here so the load is
safe either way. The neural-net and Prophet wrappers are also cloudpickle-embedded and just
need their libraries importable (handled in the install cell above).

In [ ]:
import os
# DagsHub's artifact server stalls on MLflow's multipart (parallel byte-range) downloads
# of big files; a single plain stream works. Long timeout so large files can finish.
os.environ["MLFLOW_ENABLE_MULTIPART_DOWNLOAD"] = "false"
os.environ["MLFLOW_HTTP_REQUEST_TIMEOUT"] = "600"

import re
import torch
from contextlib import contextmanager
import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.base import BaseEstimator, RegressorMixin
from src.features import CalendarFeatures, LagFeatures, DropColumns

CATEGORICAL_COLS = ["Store", "Dept"]

def signed_log1p(y):
    return np.sign(y) * np.log1p(np.abs(y))

def signed_expm1(z):
    return np.sign(z) * np.expm1(np.abs(z))

def holiday_weights(is_holiday):
    return np.where(np.asarray(is_holiday) == 1, 5.0, 1.0)

def to_lgb_frame(df, train_categories=None):
    y = df["Weekly_Sales"].to_numpy(dtype=float) if "Weekly_Sales" in df.columns else None
    X = df.drop(columns=["Weekly_Sales"]).copy() if "Weekly_Sales" in df.columns else df.copy()
    w = holiday_weights(df["IsHoliday"])
    for col in CATEGORICAL_COLS:
        if train_categories is None:
            X[col] = X[col].astype("category")
        else:
            X[col] = pd.Categorical(X[col], categories=train_categories[col])
    return X, y, w

class ToLGBFrame(BaseEstimator):
    def fit(self, X, y=None):
        X_lgb, _, _ = to_lgb_frame(X)
        self.train_categories_ = {c: X_lgb[c].cat.categories for c in CATEGORICAL_COLS}
        return self
    def transform(self, X):
        X_lgb, _, _ = to_lgb_frame(X, train_categories=self.train_categories_)
        return X_lgb

class LGBMBoosterRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, num_boost_round=100, **lgb_params):
        self.num_boost_round = num_boost_round
        self.lgb_params = lgb_params
    def fit(self, X, y, sample_weight=None):
        train_set = lgb.Dataset(X, label=y, weight=sample_weight,
                                categorical_feature=CATEGORICAL_COLS, free_raw_data=False)
        self.booster_ = lgb.train(self.lgb_params, train_set, num_boost_round=self.num_boost_round)
        return self
    def predict(self, X):
        return self.booster_.predict(X, num_iteration=self.booster_.best_iteration)

## 3. Load every candidate model from MLflow

Each row below points at one model's final full-train run. We pull the most recent matching
run, resolve the logged model (MLflow 3.x tracks it as its own entity, so we try the
`models:/<id>` and artifact-location URIs before the old `runs:/` path), load the pipeline,
and immediately predict on the raw test frame. Anything that fails to install or load is
skipped with a message instead of killing the notebook \u2014 the ensemble just uses whatever
loaded.

We also read back each model's **honest 39-week validation WMAE** that was logged at training
time. That is our best pre-submission signal for how good each model is.

In [ ]:
client = MlflowClient()

# model -> (experiment name, runName filter). The trees / Prophet / N-BEATS logged a
# final_full_train__<config> run; TFT and DLinear logged a fixed run name.
# N-BEATS goes LAST: its saved network is a big artifact the DagsHub server sometimes
# stalls on, so it must never delay the reliable models. With the long timeout above
# plus a retry it usually comes through; if it still fails it is skipped and the sweep
# below simply runs without it.
REGISTRY = {
    "LightGBM": ("lightGBM_Training",                    "tags.mlflow.runName LIKE 'final_full_train__%'"),
    "Prophet":  ("Prophet_Training",                     "tags.mlflow.runName LIKE 'final_full_train__%'"),
    "XGBoost":  ("XGBoost_Training",                     "tags.mlflow.runName LIKE 'final_full_train__%'"),
    "TFT":      ("TemporalFusionTransformer_Training",   "tags.mlflow.runName = 'tft_best'"),
    "DLinear":  ("DLinear_Training",                     "tags.mlflow.runName = 'dlinear_best'"),
    "N-BEATS":  ("NBEATS_Training",                      "tags.mlflow.runName LIKE 'final_full_train__%'"),
}

def _val_wmae(run):
    for src in (run.data.params, run.data.metrics):
        for k in ("val_wmae_of_chosen_cfg", "val_wmae"):
            if k in src:
                try: return float(src[k])
                except (TypeError, ValueError): pass
    return float("nan")

@contextmanager
def cpu_unpickle():
    """On CPU-only runtimes, map GPU-pickled tensors to CPU while unpickling."""
    if torch.cuda.is_available():
        yield; return
    orig = torch.load
    torch.load = lambda *a, **k: orig(*a, **{**k, "map_location": "cpu"})
    try:
        yield
    finally:
        torch.load = orig

def force_cpu(pipe):
    """Point every saved device hint inside a pipeline at CPU (no-op for non-torch models)."""
    for step in pipe.named_steps.values():
        if hasattr(step, "device"):
            step.device = "cpu"
        nf = getattr(step, "nf_", None)
        if nf is not None:
            for m in nf.models:
                tk = getattr(m, "trainer_kwargs", None)
                if isinstance(tk, dict):
                    tk["accelerator"] = "cpu"; tk["devices"] = 1; tk.pop("strategy", None)
        mdl = getattr(step, "model_", None)
        if mdl is not None and hasattr(mdl, "to"):
            mdl.to("cpu")
    return pipe

def robust_predict(pipe, X, max_shims=12):
    """predict() that fills library-version attribute gaps with safe empty defaults
    (our N-BEATS is univariate, so empty exog lists reproduce the old behaviour)."""
    for _ in range(max_shims):
        try:
            return np.asarray(pipe.predict(X), dtype=float)
        except AttributeError as e:
            mo = re.search(r"'(\w+)' object has no attribute '(\w+)'", str(e))
            if not mo:
                raise
            cls, attr = mo.groups()
            default = [] if attr.endswith("_list") else ({} if "vocab" in attr else None)
            targets = []
            for step in pipe.named_steps.values():
                nf = getattr(step, "nf_", None)
                if nf is not None:
                    targets = [nf] if cls == type(nf).__name__ else list(nf.models)
            if not targets:
                raise
            for t in targets:
                if not hasattr(t, attr):
                    setattr(t, attr, default)
            print(f"  shimmed {cls}.{attr} = {default!r}", flush=True)
    raise RuntimeError("still failing after max_shims attribute fixes")

def _resolve_logged_model(run):
    # MLflow 3 stores log_model() output as a standalone logged model with its own id and
    # artifact location, NOT under runs:/<id>/model (that path is empty on DagsHub). Find
    # the real id: first via run.outputs, then by searching the experiment's logged models.
    run_id = run.info.run_id
    model_id = None
    try:
        mo = run.outputs.model_outputs
        if mo:
            model_id = mo[0].model_id
    except Exception:
        pass
    if model_id is None:
        try:
            lms = client.search_logged_models(experiment_ids=[run.info.experiment_id])
            cand = [m for m in lms if getattr(m, "source_run_id", None) == run_id]
            if cand:
                model_id = cand[0].model_id
        except Exception:
            pass
    if model_id is None:
        return None, None
    return model_id, client.get_logged_model(model_id)

def load_final_pipeline(name, experiment, run_filter):
    exp = client.get_experiment_by_name(experiment)
    if exp is None:
        raise RuntimeError(f"experiment {experiment!r} not found")
    runs = client.search_runs([exp.experiment_id], filter_string=run_filter,
                              order_by=["attributes.start_time DESC"], max_results=1)
    if not runs:
        raise RuntimeError(f"no run matching {run_filter} in {experiment}")
    run = runs[0]
    model_id, lm = _resolve_logged_model(run)
    uris = []
    if lm is not None:
        uris += [f"models:/{model_id}", lm.artifact_location]
        # the path proven to work on DagsHub: register a version at the real artifact
        # location and load it back through the registry (same trick as model_inference).
        reg_name = f"walmart-ens-{name}"
        try:
            try: client.get_registered_model(reg_name)
            except Exception: client.create_registered_model(reg_name)
            mv = client.create_model_version(name=reg_name, source=lm.artifact_location,
                                             run_id=run.info.run_id)
            uris.append(f"models:/{reg_name}/{mv.version}")
        except Exception:
            pass
    uris.append(f"runs:/{run.info.run_id}/model")
    last = None
    for uri in uris:
        try:
            with cpu_unpickle():
                pipe = mlflow.sklearn.load_model(uri)
            if not torch.cuda.is_available():
                force_cpu(pipe)
            return pipe, _val_wmae(run), run.info.run_id
        except Exception as e:
            last = e
    raise RuntimeError(f"all load URIs failed; last error: {last}")

preds = {}       # model name -> np.array of test predictions
val_wmae = {}    # model name -> honest 39-week validation WMAE
for name, (experiment, run_filter) in REGISTRY.items():
    for attempt in (1, 2):   # DagsHub artifact downloads are flaky; retry once per model
        print(f"loading {name} (attempt {attempt}) ...", flush=True)
        try:
            pipe, vw, run_id = load_final_pipeline(name, experiment, run_filter)
            p = robust_predict(pipe, test)
            assert len(p) == len(test) and np.isfinite(p).all(), "bad predictions"
            preds[name] = p
            val_wmae[name] = vw
            print(f"[ok]   {name:9s} val WMAE {vw:8.1f}  run {run_id[:8]}  sample {p[:2].round(1)}", flush=True)
            break
        except Exception as e:
            if attempt == 2:
                print(f"[skip] {name:9s} -> {type(e).__name__}: {str(e)[:160]}", flush=True)

print("\nloaded models:", list(preds))

## 4. How diverse are the models?

Averaging only helps when the models **disagree** \u2014 if two models make the same errors,
their mean makes that error too. The correlation matrix below is our pre-submission read on
whether the ensemble will pay off: tree models (LightGBM, XGBoost) correlate highly with
each other, but a tree vs Prophet or a neural net (TFT, DLinear) should correlate noticeably
less. Lower cross-family correlation = more to gain from averaging. In practice Prophet is
the least correlated with LightGBM, so it is the most valuable member to add.

In [ ]:
P = pd.DataFrame(preds, index=test.index)
print("pairwise correlation of test predictions:")
print(P.corr().round(3).to_string())
print("\nhonest validation WMAE (lower = better single model):")
for k, v in sorted(val_wmae.items(), key=lambda kv: kv[1]):
    print(f"  {k:9s} {v:8.1f}")

## 5. Sweep the candidate ensembles

Each candidate is a mean over some subset of the loaded models, with two weighting schemes:
**equal** (nothing to tune, so nothing to overfit) and **inv** (each member weighted by
1/val_WMAE, so the weaker deep nets pull the mean less when they join in).

Kaggle allows unlimited late submissions and grades within seconds, so the honest way to
pick a winner is to submit every candidate and read the board. A candidate that lost a
member to a failed download collapses to whatever remains, and exact duplicates are dropped
so the same file is never submitted twice.

In [ ]:
def inv_weights(members):
    w = np.array([1.0 / val_wmae[m] for m in members])
    return w / w.sum()

def build_ensemble(members, weighting="equal"):
    members = [m for m in members if m in preds]     # keep only models that loaded
    assert members, "none of the requested members loaded"
    M = np.vstack([preds[m] for m in members])
    w = inv_weights(members) if weighting == "inv" else None
    return members, np.clip(np.average(M, axis=0, weights=w), 0, None)  # sales >= 0

def make_submission(mean_preds, path):
    sub = pd.DataFrame({
        "Id": (test["Store"].astype(str) + "_" + test["Dept"].astype(str) + "_" +
               test["Date"].dt.strftime("%Y-%m-%d")),
        "Weekly_Sales": mean_preds,
    })
    sample = pd.read_csv(os.path.join(DATA_DIR, "sampleSubmission.csv"))
    assert len(sub) == len(sample) and set(sub["Id"]) == set(sample["Id"]), "ids do not match sample"
    sub.to_csv(path, index=False)
    return sub

# tag -> (members, weighting), strongest hypotheses first
CANDIDATES = [
    ("lgb_prophet",        ["LightGBM", "Prophet"],                       "equal"),
    ("lgb_prophet_nbeats", ["LightGBM", "Prophet", "N-BEATS"],            "equal"),
    ("lgb_prophet_xgb",    ["LightGBM", "Prophet", "XGBoost"],            "equal"),
    ("top4",               ["LightGBM", "Prophet", "N-BEATS", "XGBoost"], "equal"),
    ("all_equal",          list(preds),                                   "equal"),
    ("all_invwmae",        list(preds),                                   "inv"),
]

# drop candidates that collapse to a duplicate after removing failed models
seen, SWEEP = set(), []
for tag, members, weighting in CANDIDATES:
    used = tuple(m for m in members if m in preds)
    key = (used, weighting if len(used) > 1 else "equal")
    if len(used) >= 2 and key not in seen:
        seen.add(key); SWEEP.append((tag, list(used), weighting))

for tag, used, weighting in SWEEP:
    print(f"{tag:20s} {weighting:5s} mean{tuple(used)}")

In [ ]:
# build and submit every candidate; Kaggle grades each within seconds
for tag, members, weighting in SWEEP:
    used, mean_preds = build_ensemble(members, weighting)
    path = os.path.join(DATA_DIR, f"sub_{tag}.csv")
    make_submission(mean_preds, path)
    msg = f"ensemble {tag}: {weighting} mean of {'+'.join(used)}"
    subprocess.run(["kaggle", "competitions", "submit",
                    "-c", KAGGLE_COMP, "-f", path, "-m", msg], check=True)
    print("submitted", tag, "->", used, flush=True)

In [ ]:
# wait for grading, then read the board. Rerun this cell until nothing says PENDING,
# then the row with the LOWEST privateScore is our final model.
import time; time.sleep(60)
r = subprocess.run(["kaggle", "competitions", "submissions", "-c", KAGGLE_COMP],
                   capture_output=True, text=True)
print(r.stdout)